# The motorways of a country

This notebook loads every motorway of Brazil, with every point of each road, and computes a route on it. The data is the OpenStreetMap extract of Brazil, a file of about 2.1 GB under the [Open Database License (ODbL 1.0)](https://opendatacommons.org/licenses/odbl/1-0/). After the download, the run takes about two minutes and about 2 GB of memory.

An OpenStreetMap **way** is a line of a road, of a railway or of another feature. Its tag `highway` gives the kind of road: `motorway`, `primary`, `residential` and others. The **shape** of an edge is the list of points that draw it on a map between its two nodes.

1. Download the extract of Brazil.
2. Choose the motorways.
3. Load them, with their shapes.
4. See why the load keeps one component only.
5. Compute a route and draw it.
6. Save the network to a file.

With a slow connection, use the extract of the south-east of Brazil: in the first cell of section 1, replace `brazil` by `brazil/sudeste` (about 0.9 GB). The steps are the same, and São Paulo and Rio de Janeiro are both in this extract.

GitHub does not show the maps of a notebook. To see them, run the notebook, or open it in [nbviewer](https://nbviewer.org/github/connor-makowski/scgraph/blob/main/examples/motorways_of_a_country.ipynb).

## Setup

`load_from_osm` reads OpenStreetMap files with `osmium`, which the extra `osm` of scgraph installs. The map needs [folium](https://python-visualization.github.io/folium/).

In [1]:
# Until scgraph 4.0 is on PyPI, install it from the `dev` branch:
# %pip install "scgraph[osm] @ git+https://github.com/connor-makowski/scgraph@dev" folium
import logging
import tempfile
from itertools import takewhile
from pathlib import Path

import folium

from scgraph import GeoGraph
from scgraph.helpers.osm import get_way_direction
from scgraph.pipelines import download_file

## 1. Download the extract

[Geofabrik](https://download.geofabrik.de) publishes an extract of OpenStreetMap for each country, every day. The extract of Brazil holds the data inside the border of the country.

The next cell downloads the extract into the temporary folder of your system. It is about 2.1 GB. A later run finds the file there and downloads nothing. The data is © OpenStreetMap contributors, under the ODbL 1.0: give this credit with your results.

In [2]:
logging.basicConfig(format="%(message)s")
logging.getLogger("scgraph.pipelines").setLevel(logging.INFO)  # shows the progress of the download

EXTRACT_URL = "https://download.geofabrik.de/south-america/brazil-latest.osm.pbf"
data_folder = Path(tempfile.gettempdir()) / "scgraph_examples"
data_folder.mkdir(exist_ok=True)
extract_file = data_folder / EXTRACT_URL.split("/")[-1]
if not extract_file.exists():
    # The file gets its name at the end only, so a stopped download is never read
    partial_file = data_folder / "download.part"
    download_file(EXTRACT_URL, partial_file)
    partial_file.rename(extract_file)
print(f"{extract_file.name}: {extract_file.stat().st_size / 1e9:.1f} GB")

brazil-latest.osm.pbf: 2.1 GB


## 2. Choose the motorways

`select` chooses the ways of the file. It receives the tags of a way. It returns None to leave the way out, or a pair: the direction and the class of the way.

- The **direction** is 1 or -1 for a one-way road, and 0 for a road in both directions. `get_way_direction` reads it from the tags.
- The **class** of an edge is a name. Here it is the `highway` value of the way. A `motorway_link` is a ramp that joins a motorway to another road.

In [3]:
MOTORWAYS = ["motorway", "motorway_link"]


def select_motorways(tags):
    """The direction and the class of a motorway, and None for any other way."""
    if tags.get("highway") not in MOTORWAYS:
        return None
    return get_way_direction(tags), tags["highway"]

## 3. Load the motorways

`load_from_osm` reads the file and builds a geograph from the ways that `select` keeps. A node is a point where two kept ways meet, or where a way ends. The load also cuts a long way, so that two nodes are at most about 2 km apart. Every other point of a way stays in the shape of its edge.

The load takes about a minute. Look at the number of shape points: it is several times the number of nodes.

In [4]:
motorways = GeoGraph.load_from_osm(extract_file, select=select_motorways)

print(f"{len(motorways.nodes):,} nodes")
for edge_class in MOTORWAYS:
    edges = motorways.get_edges(edge_class=edge_class)
    km = sum(edge["length"] for edge in edges)
    print(f"{edge_class:<15}{len(edges):>7,} edges{km:>8,.0f} km")
shape_points = sum(len(shape) for row in motorways.intermediate_nodes for shape in row.values())
print(f"{shape_points:,} shape points")

38,720 nodes
motorway        32,988 edges  15,801 km
motorway_link   10,670 edges   1,742 km


207,775 shape points


A motorway has two carriageways. OpenStreetMap draws each one as a one-way way, so the km above count both.

## 4. Why the load keeps one component only

In OpenStreetMap, few roads of Brazil have the tag `motorway`: most of the large highways have the tag `trunk`. So the motorways of Brazil form several groups of roads that do not touch.

A **strongly connected component** is a set of nodes where every node can reach every other node by a route. A route cannot go from one component to another.

By default, `load_from_osm` keeps the largest component only. So a place always snaps to a node that a route can reach. `min_component_nodes=0` keeps every component. A place can then snap to a node of a small component, and a route to a place on another component raises `PathNotFoundError`.

You can check both facts with this notebook. Load again with `min_component_nodes=0`, and compare `len(motorways.nodes)` with the number above. Then add `"trunk"` and `"trunk_link"` to `MOTORWAYS`, to see the highways that join the groups.

## 5. A route from São Paulo to Rio de Janeiro

`output_path=True` adds `path` to the result: the node ids of the route. `output_legs=True` adds the legs. An **edge mode** is the kind of transport of an edge: `road` here, and `off_graph` for the straight line between a place and its closest node. A leg is a part of the route on one edge mode, and `class_lengths` gives its km in each class.

Look at the number of points of `coordinate_path`. It holds the nodes and the shapes, so it is much larger than the number of nodes.

In [5]:
SAO_PAULO = {"latitude": -23.55, "longitude": -46.63}
RIO_DE_JANEIRO = {"latitude": -22.90, "longitude": -43.20}

route = motorways.get_shortest_path(
    origin_node=SAO_PAULO, destination_node=RIO_DE_JANEIRO, output_path=True, output_legs=True
)
print(f"{route['length']:,.1f} km")
print(f"{len(route['path']):,} nodes and {len(route['coordinate_path']):,} points")
for leg in route["legs"]:
    km_by_class = ", ".join(f"{name} {km:,.1f} km" for name, km in leg["class_lengths"].items())
    print(f"    {leg['edge_mode']:<10}{leg['length']:>7,.1f} km  {km_by_class}")

429.3 km
1,071 nodes and 5,478 points
    off_graph     3.4 km  
    road        423.7 km  motorway 421.8 km, motorway_link 1.9 km
    off_graph     2.2 km  


The first leg and the last leg have the mode `off_graph`: they are the straight lines between the two cities and their closest nodes. The leg on the road gives its km on `motorway` and on `motorway_link`.

The map shows the start of the route, in São Paulo. The route is in black. The other motorways of the area are in orange for `motorway` and in blue for `motorway_link`. Each line goes through every point of its shape, so it follows the curves of its road and of its ramps.

A map of the whole route would hold thousands of points, too many for the saved notebook: so the map shows one area only.

In [6]:
CLASS_COLORS = {"motorway": "#eb6834", "motorway_link": "#0072B2"}
SAO_PAULO_AREA = (-23.56, -46.66, -23.50, -46.56)  # south, west, north, east
south, west, north, east = SAO_PAULO_AREA

route_map = folium.Map(location=[-23.53, -46.61], zoom_start=14)
for node in motorways.get_nodes(bbox=SAO_PAULO_AREA):
    origin_id = node["node_id"]
    for edge in motorways.get_edges(origin_id=origin_id):
        destination_id = edge["destination_id"]
        shape = motorways.intermediate_nodes[origin_id].get(destination_id, [])
        points = [motorways.nodes[origin_id], *shape, motorways.nodes[destination_id]]
        folium.PolyLine(points, color=CLASS_COLORS[edge["edge_class"]], weight=2).add_to(route_map)
# The points of the route from its origin to the first point outside the area
route_start = list(
    takewhile(lambda point: south <= point[0] <= north and west <= point[1] <= east, route["coordinate_path"])
)
folium.PolyLine(route_start, color="#0b0b0b", weight=4).add_to(route_map)
route_map

## 6. Save the network

`save_as_scgraph` writes the geograph to one file, with its classes and its shapes. `load_from_scgraph` reads it back in a moment, without the extract. The route on the loaded network has the same length.

In [7]:
network_file = data_folder / "brazil_motorways.scgraph"
motorways.save_as_scgraph(network_file)
print(f"{network_file.name}: {network_file.stat().st_size / 1e6:.1f} MB")

motorways_again = GeoGraph.load_from_scgraph(network_file)
route_again = motorways_again.get_shortest_path(origin_node=SAO_PAULO, destination_node=RIO_DE_JANEIRO)
print(f"{len(motorways_again.nodes):,} nodes, route of {route_again['length']:,.1f} km")

brazil_motorways.scgraph: 3.2 MB
38,720 nodes, route of 429.3 km


## Other ways to get the roads of Brazil

| What you need | Call | Shapes |
|---|---|---|
| Every motorway, with every point (this notebook) | `GeoGraph.load_from_osm(extract_file, select=select_motorways)` | Every point of the ways |
| Every main road (`motorway` to `primary`), with railways, sea lanes, ports and airports | `GeoGraph.build_geograph("osm_region_multimodal", region="south-america/brazil")`, which downloads the same extract and needs more memory | Simplified to within 50 m |
| The main roads of South America, ready to load | `GeoGraph.load_geograph("south_america", node_modes=["road"])` | Simplified to within 50 m |

`select` gives a class from `scgraph.pipelines.CLASSES`, which has no class for a road below `primary`. So a `secondary` road cannot keep its own class in this geograph.

## What to remember

- `GeoGraph.load_from_osm(file, select=...)` loads the ways that `select` keeps, with every point of their shapes. It needs `pip install scgraph[osm]`.
- `select` returns the direction and the class of a way, or None to leave it out.
- By default, the load keeps the largest strongly connected component, where every node can reach every other node.
- `save_as_scgraph` writes the network to a file, and `load_from_scgraph` reads it back without the extract.

The notebook *The streets of a city* loads every street of Boston, with its turn restrictions.